In [ ]:
from crewai import LLM, Agent , Task , Crew, Process
from dotenv import load_dotenv
load_dotenv(override= True)
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, DirectoryReadTool, FileWriterTool, FileReadTool
from datetime import datetime


In [ ]:
llm = LLM(model = "gemini/gemini-2.5-flash", temperature = 0.7)

Agents
1. Head of Marketing
2. Social Media Content Creater
3. Blog Writer
4. SEO Specialist

In [ ]:
head_of_marketing = Agent(
    role = "Head of Marketing",
    goal = "Lead the marketing team to achieve strategic goals and drive brand growth, by researching, planning and buildig a marketing strategy.",
    backstory = '''You are an experienced marketing professional with a track record of leading successful marketing campaigns. 
                You have a deep understanding of market trends and consumer behavior, and you are skilled in developing and executing marketing strategies 
                 that drive brand growth.''',

    llm = llm,
    tools=[
                SerperDevTool(),
                ScrapeWebsiteTool(),
                DirectoryReadTool('resources/drafts'),
                FileWriterTool(),
                FileReadTool()
            ],
    reasoning=True,
    inject_date=True,
    allow_delegation=True,
    max_rpm=1
)


social_media_content = Agent(
    role = "Content Creator for social media",
    goal = "Create engaging and innovative content that resonates with the target audience, by researching, planning and building a content strategy.",
    backstory = '''You are a creative professional with a passion for storytelling and content creation. 
                You have a knack for producing high-quality content that captures attention and drives engagement through reels, posts and email campaigns.''',
    
    tools=[
        SerperDevTool(),
        ScrapeWebsiteTool(),
        DirectoryReadTool('resources/drafts/blogs'),
        FileWriterTool(),
        FileReadTool()
    ],
    inject_date=True,
    allow_delegation=True,
    max_iter=1,
    max_rpm=1,
    llm = llm
)


blog_writer = Agent(
    role = 'Content Writer for blogs',
    goal = '''Write compelling and persuasive blogs that aligns with the brand's voice and messaging, by researching and planning based on a content strategy.''',
    backstory = '''You are a skilled writer with a talent for crafting engaging and informative content.
                 You have a strong understanding of content marketing, and you are able to produce high-quality blogs that drive traffic and engagement.''',
    llm = llm,
    tools=[
                SerperDevTool(),
                ScrapeWebsiteTool(),
                DirectoryReadTool('resources/drafts'),
                FileWriterTool(),
                FileReadTool()
            ],
    inject_date=True,
    allow_delegation=True,
    max_iter=1,
    max_rpm=1
)


seo_specialist = Agent(
    role = "SEO Specialist" ,
    goal = "Optimize the blogs and content for search engines to improve visibility and drive organic traffic.",
    backstory = '''You are an SEO expert with a deep understanding of search engine algorithms and ranking factors.
                  You have a proven track record of improving website visibility and driving organic traffic through effective SEO strategies. 
                  Store the SEO optimized blogs in markdown files.''',
    llm = llm,
        tools=[
                SerperDevTool(),
                ScrapeWebsiteTool(),
                DirectoryReadTool('resources/drafts'),
                FileWriterTool(),
                FileReadTool()
            ],
    inject_date=True,
    allow_delegation=True,
    max_iter=1,
    max_rpm=1
)

Task
1. Market Research
2. Prepare market strategy
3. Create Content Calender
4. Prepare Post Draft
5. Prepare Script for reels
6. Content Research for Blogs
7. Draft Blogs
8. SEO Optimization



In [ ]:
market_research= Task(
    description = '''Conduct market research to identify trends, opportunities, and challenges in the industry. 
    Focus on understanding customer needs, competitor strategies, and market dynamics to inform marketing decisions.
    <Product Name>{product_name}</Product Name>
    <Product Description>{product_description}</Product Description>
    <Target Audience>{target_audience}</Target Audience>
    <Budget>{budget}</Budget>

    Current date: {current_date}''',
    expected_output = '''A comprehensive market research report that includes:
    - Analysis of market trends and opportunities
    - Competitor analysis
    - Customer insights and needs
    - Recommendations for marketing strategies''',
    agent = head_of_marketing
)


prepare_marketing_strategy = Task(
    description ='''
    Develop a marketing strategy that aligns with the company's goals and objectives. 
    The strategy should include target audience segmentation, positioning, messaging, and marketing channels.
    <Product Name>{product_name}</Product Name>
    <Product Description>{product_description}</Product Description>
    <Target Audience>{target_audience}</Target Audience>

    Current date: {current_date}
    Refer to the market research report for insights and building the strategy.''',

    expected_output= 
    """A detailed marketing strategy document that includes:
    - Target audience segmentation
    - Required budget
    - Recommended marketing channels
    - Plan of action for the week containing 3 blogs to write, 2 reels to film, and 5 social media posts to create
    - Key performance indicators (KPIs) to measure success
    Store the marketing strategy in 'resources/drafts/marketing_strategy.md'.""",
    agent = head_of_marketing)

create_content_calendar = Task(
  description=
    """Create a content calendar that outlines the topics, formats, and publishing schedule for the next one week.
      The calendar should align with the marketing strategy and include key themes and campaigns.
    <Product Name>{product_name}</Product Name>
    <Product Description>{product_description}</Product Description>
    <Target Audience>{target_audience}</Target Audience>

    Current date: {current_date}""",

   expected_output= 
    """A content calendar that includes:
    - Topics and formats for each piece of content specified in the marketing strategy
    - Publishing schedule
    - Key themes and campaigns
    Store the content calendar in 'resources/drafts/content_calendar.md'.""",
    agent = social_media_content
    )

prepare_post_drafts = Task(
  description = 
  """  Prepare drafts for social media posts and email campaigns based on the content calendar.
    Ensure that the drafts are engaging, aligned with the brand voice, and optimized for each platform.
    <Product Name>{product_name}</Product Name>
    <Product Description>{product_description}</Product Description>
    <Target Audience>{target_audience}</Target Audience>

    Current date: {current_date}
""",
  expected_output=
    """Drafts for:
    - Social media posts: LinkedIn, Twitter, Instagram
    - Email campaigns
    Each draft should be ready for review and feedback. Save the drafts in 'resources/drafts/posts/' folder in markdown format.""",
    agent = social_media_content,
    )

prepare_scripts_for_reels = Task(
    description = """
    Prepare scripts for Instagram reels that align with the content calendar and marketing strategy.
      The scripts should be engaging, concise, and tailored to the target audience.
    <Product Name>{product_name}</Product Name>
    <Product Description>{product_description}</Product Description>
    <Target Audience>{target_audience}</Target Audience>

    Current date: {current_date}""",

    expected_output = """
    Scripts for Instagram reels that include:
    - Engaging hooks
    - Key messages
    - Call to action
    Each script should be ready for filming and editing. Save the scripts in 'resources/drafts/reels/' folder in markdown format. 
    And assign tasks to the relevant team members.""",
    agent = social_media_content
    )

content_research_for_blogs = Task (
    description = """
        Conduct research for blog topics that align with the content calendar and marketing strategy. 
        The research should include keyword analysis, competitor blogs, and industry trends.
        <Product Name>{product_name}</Product Name>
        <Product Description>{product_description}</Product Description>
        <Target Audience>{target_audience}</Target Audience>

        Current date: {current_date}""",

    expected_output = """
        A research report that includes:
        - Keyword analysis
        - Competitor blog insights
        - Industry trends
        - Suggested blog topics and outlines""",
        agent = blog_writer
)

draft_blogs =  Task(
  description = """
    Draft blogs based on the research report and content calendar. Each blog should be well-structured, engaging, and optimized for SEO.
    <Product Name>{product_name}</Product Name>
    <Product Description>{product_description}</Product Description>
    <Target Audience>{target_audience}</Target Audience>

    Current date: {current_date}""",

  expected_output = """
    Drafts for blogs that include:
    - Engaging introductions
    - Well-structured content
    - SEO optimization
    - Ready for review and feedback
    Each draft should be ready for review and saved in 'resources/drafts/blogs/' folder in markdown format.""",
    agent = blog_writer
)

seo_optimization = Task(
    description = """
        Optimize the drafted blogs for search engines to improve visibility and drive organic traffic. This includes keyword optimization, meta tags, and internal linking.
        <Product Name>{product_name}</Product Name>
        <Product Description>{product_description}</Product Description>
        <Target Audience>{target_audience}</Target Audience>

        Current date: {current_date}""",

    expected_output = """
        SEO-optimized blogs that include:
        - Keyword-rich titles and headings
        - Meta descriptions
        - Internal links
        - Ready for final review and publication""",
    agent = seo_specialist)

In [ ]:
crew =  Crew(
    agents = [head_of_marketing, social_media_content, blog_writer, seo_specialist],
    tasks = [market_research,prepare_marketing_strategy,create_content_calendar,prepare_post_drafts,prepare_scripts_for_reels,
            content_research_for_blogs, draft_blogs, seo_optimization],
    process=Process.sequential,
    verbose=True,
    planning=True,
    planning_llm=llm,
    max_rpm=1
)

In [ ]:
result = crew.kickoff(inputs= {
        "product_name": "AI Powered Excel Automation Tool",
        "target_audience": "Small and Medium Enterprises (SMEs)",
        "product_description": "A tool that automates repetitive tasks in Excel using AI, saving time and reducing errors.",
        "budget": "Rs. 50,000",
        "current_date": datetime.now().strftime("%Y-%m-%d"),
    }
                    )

In [ ]:
print("Marketing crew has been successfully created and run.")